# learner

> `DecisionTrainer`, a `transformers.Trainer` with two overrides, and `Learner`, fastai's verbs over it: `lr_find`, `fit`, `fit_one_cycle`, `freeze`, `calibrate`.

In [ ]:
#| default_exp learner

In [ ]:
#| export
import copy, dataclasses, math, os, tempfile, warnings
from pathlib import Path

import numpy as np
import torch
from fastcore.basics import patch, store_attr
from transformers import Trainer, TrainingArguments, TrainerCallback

from sysone.core import *
from sysone.template import *
from sysone.data import *
from sysone.models import *
from sysone.losses import *
from sysone.metrics import Preds, as_metric

In [ ]:
#| hide
from fastcore.test import test_eq, test_close, test_fail, test_ne
import json, logging
logging.getLogger("transformers").setLevel(logging.ERROR)
import datasets, transformers; datasets.disable_progress_bars(); transformers.utils.logging.disable_progress_bar()
from transformers import get_scheduler
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 9,
                     "axes.titlesize": 9, "legend.fontsize": 8, "legend.frameon": False})

Training is `transformers.Trainer` with two overrides, the loss and the optimizer groups, plus what Laya's notebook does by hand: the RLCD loss, discriminative learning rates, the σ schedule and a calibration pass on held-out rows. The training loop, mixed precision, gradient accumulation, checkpointing, distributed training and the reporters are the Trainer's, untouched. On top sit fastai's verbs: `lr_find`, `fit`, `fit_one_cycle`, `freeze`, `unfreeze`, `freeze_to`.

The fixtures again: twenty cases and the tiny ModernBERT.

In [ ]:
FIX = "fixtures/tiny-modernbert"
data = TypedDecisions.from_json("fixtures/typed_decisions_tiny.jsonl", valid=0.25, calib=0.2)
spec = EncoderSpec.from_pretrained(FIX)
data.bind(spec.tokenizer, spec.template)
data

## How a fit runs

One call to `fit_one_cycle`, from the verb to the temperatures:

```{mermaid}
sequenceDiagram
    participant U as you
    participant L as Learner
    participant T as DecisionTrainer (a transformers.Trainer)
    participant M as EncoderDecisionModel
    participant C as callbacks
    U->>L: fit_one_cycle(epochs, lr)
    L->>T: _run: schedule arguments set in place, optimizer reset
    T->>T: create_optimizer: head and encoder groups, no decay on biases and norms
    loop every epoch
        T->>C: on_epoch_begin: SigmaSchedule sets σ, OptionShuffle sets the epoch
        loop every batch
            T->>M: collated batch: input_ids, marker_pos, marker_mask, qtype
            M-->>T: logits (batch, K)
            T->>T: compute_loss (RLCD in training, soft CE in evaluation), backward, clip at 1, step
        end
        T->>C: on_evaluate: Recorder keeps the metrics
    end
    T->>C: on_train_end: CalibrationCallback
    C->>L: calibrate(): temperatures fitted on the calib split
    L-->>U: the last epoch's metrics
```

Everything in the loop is the Trainer's own; sysone supplies the loss, the optimizer groups, two schedules and five callbacks.

## DecisionTrainer

`compute_loss` pops the targets, runs the model and applies the loss; in evaluation it uses soft cross-entropy, since RLCD's noise would make the evaluation loss jitter. `prediction_step` returns the logits and, packed into one tensor, what the metrics need per row: the target, the marker mask, the type and the label. The Trainer pads everything it gathers across batches with -100, so the packing puts the options on the axis it pads and the mask tells real options from padding. Evaluation reads rows in order, so predictions line up with the dataset.

In [ ]:
#| export
def pack_labels(b) -> torch.Tensor:
    "Target, marker mask, question type, label and answerability of a batch as one (batch, K, 5) tensor"
    K = b["marker_mask"].shape[1]
    tgt = b["target"].float() if "target" in b else torch.zeros_like(b["marker_mask"], dtype=torch.float)
    rep = lambda v: v.float()[:, None].expand(-1, K)
    lab = b["label"] if "label" in b else torch.full_like(b["qtype"], -1)
    ans = b["answerable"] if "answerable" in b else torch.full_like(b["qtype"], -1)
    return torch.stack([tgt, b["marker_mask"].float(), rep(b["qtype"]), rep(lab), rep(ans)], -1)

def unpack_labels(packed):
    "targets, masks, qtypes, labels from `pack_labels` output gathered by the Trainer (padded with -100)"
    packed = np.asarray(packed)
    mask = packed[..., 1] == 1
    return np.where(mask, packed[..., 0], 0.0), mask, packed[:, 0, 2].astype(int), packed[:, 0, 3].astype(int)

def unpack_answerable(packed):
    "Each row's answerability from `pack_labels` output (1 answerable, 0 not, -1 no gold)"
    packed = np.asarray(packed)
    return packed[:, 0, 4].astype(int) if packed.shape[-1] > 4 else np.ones(len(packed), dtype=int)

In [ ]:
b = data.collator([data.rows()[i] for i in range(4)])
tg, mk, qt, lb = unpack_labels(np.pad(pack_labels(b).numpy(), ((0, 0), (0, 2), (0, 0)), constant_values=-100))
test_eq(mk, np.pad(b["marker_mask"].numpy(), ((0, 0), (0, 2))))
test_eq((qt, lb), (b["qtype"].numpy(), b["label"].numpy()))

The packing, for two evaluation batches with different option counts: the Trainer joins them with `nested_concat`, padding the options axis with −100, and the mask channel is what tells real options from padding when `unpack_labels` reads them back.

![Two packed batches joined by the Trainer, padded with -100](images/pack_labels.svg)

*Drawn with `pack_labels` and the Trainer's `nested_concat` in [figures](90_figures.ipynb#labels-packed-for-the-trainer).*

In [ ]:
#| export
class DecisionTrainer(Trainer):
    "A `Trainer` with the decision loss, head and encoder learning rates, and an optional LR-range or one-cycle schedule"
    def __init__(self, *args, loss_fn=None, lr_head:float|None=None, lr_encoder:float|None=None, act_weight:float=0.5, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_fn, self.lr_head, self.lr_encoder, self.act_weight = loss_fn or soft_ce, lr_head, lr_encoder, act_weight
        self.model_accepts_loss_kwargs = False   # the loss is a mean; gradient accumulation must scale it
        self.schedule = None                     # None (the args' scheduler), ("range", start, end) or ("onecycle", pct_start)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        inputs = dict(inputs)
        target, label = inputs.pop("target", None), inputs.pop("label", None)
        out = model(**inputs)
        logits, act = out if isinstance(out, tuple) else (out, None)
        fn = self.loss_fn if model.training else soft_ce
        if target is None: loss = logits.sum() * 0
        else:
            rows = target.sum(-1) > 0                   # an unanswerable row teaches the act head only
            loss = fn(logits[rows], target[rows], inputs["marker_mask"][rows], inputs["qtype"][rows]) if bool(rows.any()) else logits.sum() * 0
            if act is not None and label is not None:
                loss = loss + self.act_weight * act_loss(act, logits, label, inputs.get("answerable", torch.ones_like(label)))
        return (loss, {"logits": logits, "act": act}) if return_outputs else loss

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        with torch.no_grad(), self.compute_loss_context_manager():
            loss, out = self.compute_loss(model, inputs, return_outputs=True)
        if prediction_loss_only: return loss.detach(), None, None
        preds = out["logits"].detach().float()
        return loss.detach(), (preds, out["act"].detach().float()) if out["act"] is not None else preds, pack_labels(inputs)

    def _get_eval_sampler(self, eval_dataset):
        # predictions come back in dataset order (the Trainer would group evaluation rows by length too)
        if eval_dataset is None or not hasattr(eval_dataset, "__len__"): return None
        return torch.utils.data.SequentialSampler(eval_dataset) if self.args.world_size <= 1 else None

`create_optimizer` builds four groups, head and encoder (or its adapters), each split by the Trainer's own no-decay rule for biases and norms. The head's groups come first, so the learning rate the Trainer logs is the head's. `create_scheduler` adds two schedules the Trainer lacks: the exponential ramp of the LR range test and torch's exact `OneCycleLR`; both scale every group, so the encoder keeps its ratio to the head.

In [ ]:
#| export
@patch
def create_optimizer(self:DecisionTrainer, model=None):
    opt_model = self.model if model is None else model
    if self.optimizer is None:
        decay = set(self.get_decay_parameter_names(opt_model))
        lr = {"head": self.lr_head or self.args.learning_rate, "encoder": self.lr_encoder or self.args.learning_rate}
        buckets = {}
        for n, p in opt_model.named_parameters():
            if p.requires_grad: buckets.setdefault(("encoder" if n.startswith("encoder.") else "head", n in decay), []).append(p)
        groups = [{"params": ps, "lr": lr[part], "weight_decay": self.args.weight_decay if d else 0.0,
                   "name": f"{part}/{'decay' if d else 'no_decay'}"}
                  for (part, d), ps in sorted(buckets.items(), key=lambda kv: (kv[0][0] != "head", not kv[0][1]))]
        cls, kw = self.get_optimizer_cls_and_kwargs(self.args, opt_model)
        self.optimizer = cls(groups, **kw)
    return self.optimizer

@patch
def create_scheduler(self:DecisionTrainer, num_training_steps:int, optimizer=None):
    if self.schedule is None or self.lr_scheduler is not None:
        return Trainer.create_scheduler(self, num_training_steps, optimizer)
    optimizer = self.optimizer if optimizer is None else optimizer
    if self.schedule[0] == "range":
        start, end = self.schedule[1:]
        gamma = (end / start) ** (1.0 / max(1, num_training_steps - 1))
        scale = start / (self.lr_head or self.args.learning_rate)
        self.lr_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s: scale * gamma ** s)
    else:
        self.lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=[g["lr"] for g in optimizer.param_groups], total_steps=max(2, num_training_steps),
            pct_start=self.schedule[1], anneal_strategy="cos", cycle_momentum=False)
    self._created_lr_scheduler = True
    return self.lr_scheduler

## TrainingArguments

`TrainingArguments` is used as is. `training_args` fills in what decision training wants: no checkpoints, no reporters, the column names our collator reads, length-grouped batches on the precomputed `n_tokens`, clipping at 1 and weight decay 0.01, and the arguments of the current transformers version where they were renamed (`warmup_steps` as a ratio replaced `warmup_ratio`, `train_sampling_strategy` replaced `group_by_length`).

In [ ]:
#| export
_TA_FIELDS = {f.name for f in dataclasses.fields(TrainingArguments)}

def _compat(kw:dict) -> dict:
    "Translate TrainingArguments keywords between transformers versions"
    kw = dict(kw)
    if "warmup_steps" in kw and "warmup_ratio" in _TA_FIELDS and isinstance(kw["warmup_steps"], float) and kw["warmup_steps"] < 1:
        kw["warmup_ratio"] = kw.pop("warmup_steps")
    if kw.get("train_sampling_strategy") == "group_by_length" and "train_sampling_strategy" not in _TA_FIELDS:
        kw.pop("train_sampling_strategy"); kw["group_by_length"] = True
    if "eval_strategy" in kw and "eval_strategy" not in _TA_FIELDS: kw["evaluation_strategy"] = kw.pop("eval_strategy")
    unknown = set(kw) - _TA_FIELDS
    if unknown: raise TypeError(f"TrainingArguments does not take {sorted(unknown)} in this transformers version")
    return kw

def training_args(output_dir, **kw) -> TrainingArguments:
    "`TrainingArguments` with sysone's defaults, overridable by keyword"
    base = dict(output_dir=str(output_dir), report_to="none", save_strategy="no", logging_strategy="steps", logging_steps=10,
                remove_unused_columns=False, label_names=["target"], max_grad_norm=1.0, weight_decay=0.01,
                lr_scheduler_type="cosine", warmup_steps=0.1, train_sampling_strategy="group_by_length",
                length_column_name="n_tokens", per_device_train_batch_size=8, per_device_eval_batch_size=16,
                dataloader_pin_memory=torch.cuda.is_available(), seed=0)
    return TrainingArguments(**_compat(base | kw))

### Hardware presets

Presets are plain dictionaries of `TrainingArguments`, to print and edit. `macbook` is MPS in fp32 with small batches, meant for head-only runs with the cache (bf16 runs on MPS but costs accuracy on some encoders); `t4` and `t4x2` are fp16 with gradient checkpointing and 8 × 4 accumulation, the notebook's shape (for `t4x2`, launch with `torchrun --nproc_per_node=2`); `l4` and `a100` are bf16. `SYSONE_PRESET` picks one from the environment, which is how the cloud wrappers choose the preset of the machine they rent.

In [ ]:
#| export
PRESETS = {
    "cpu":     dict(use_cpu=True, per_device_train_batch_size=8, per_device_eval_batch_size=16),
    "macbook": dict(per_device_train_batch_size=4, per_device_eval_batch_size=8, dataloader_pin_memory=False),
    "t4":      dict(fp16=True, gradient_checkpointing=True, per_device_train_batch_size=8, gradient_accumulation_steps=4,
                    per_device_eval_batch_size=16),
    "t4x2":    dict(fp16=True, gradient_checkpointing=True, per_device_train_batch_size=8, gradient_accumulation_steps=4,
                    per_device_eval_batch_size=16, ddp_find_unused_parameters=False),
    "l4":      dict(bf16=True, gradient_checkpointing=True, per_device_train_batch_size=16, gradient_accumulation_steps=2,
                    per_device_eval_batch_size=32),
    "a100":    dict(bf16=True, per_device_train_batch_size=32, per_device_eval_batch_size=64),
}

def default_preset() -> str:
    "`SYSONE_PRESET`, else a guess from the hardware"
    if os.environ.get("SYSONE_PRESET"): return os.environ["SYSONE_PRESET"]
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        if "T4" in name: return "t4x2" if torch.cuda.device_count() > 1 else "t4"
        if "L4" in name: return "l4"
        return "a100"
    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available(): return "macbook"
    return "cpu"

In [ ]:
args = training_args(tempfile.mkdtemp(), **PRESETS["cpu"])
test_eq((args.device.type, args.save_strategy, args.label_names, args.remove_unused_columns), ("cpu", "no", ["target"], False))
PRESETS["t4"]

## Callbacks

Five small callbacks do what the notebook did by hand. `SigmaSchedule` anneals RLCD's σ once per epoch; `OptionShuffle` tells the `Shuffle` transforms which epoch it is; `Recorder` keeps the losses and the evaluation metrics; `LRFinder` records the loss at every step of the range test and stops the run when the smoothed loss passes four times its best; `CalibrationCallback` fits the temperatures on the held-out slice when training ends, however the training was started.

In [ ]:
#| export
def _epoch(state) -> int: return int(round(state.epoch or 0))

class SigmaSchedule(TrainerCallback):
    "Anneal RLCD's σ once per epoch"
    def __init__(self, loss): self.loss = loss
    def on_epoch_begin(self, args, state, control, **kw):
        if hasattr(self.loss, "set_epoch"): self.loss.set_epoch(_epoch(state), math.ceil(state.num_train_epochs or 1))

class OptionShuffle(TrainerCallback):
    "Give every `Shuffle` transform the current epoch, so option order changes between epochs"
    def __init__(self, tfms): self.tfms = [t for t in tfms if hasattr(t, "set_epoch")]
    def on_epoch_begin(self, args, state, control, **kw):
        for t in self.tfms: t.set_epoch(_epoch(state))

class Recorder(TrainerCallback):
    "Training losses and evaluation metrics, per step and per epoch"
    def __init__(self): self.losses, self.lrs, self.metrics = [], [], []
    def on_log(self, args, state, control, logs=None, **kw):
        if logs and "loss" in logs: self.losses.append(logs["loss"]); self.lrs.append(logs.get("learning_rate"))
    def on_evaluate(self, args, state, control, metrics=None, **kw):
        if metrics: self.metrics.append({"epoch": state.epoch, **{k.removeprefix("eval_"): v for k, v in metrics.items()}})

class CalibrationCallback(TrainerCallback):
    "Fit the learner's temperatures on its calib split when training ends"
    def __init__(self, learner): self.learner, self.active = learner, True
    def on_train_end(self, args, state, control, **kw):
        if self.active and self.learner.data.cases.get("calib") is not None: self.learner.calibrate()

class LRFinder(TrainerCallback):
    "Record (lr, loss) every step and stop when the smoothed loss diverges"
    def __init__(self, factor:float=4.0, beta:float=0.98, stop_div:bool=True):
        store_attr(); self.lrs, self.losses, self.avg, self.best, self.n = [], [], 0.0, math.inf, 0
    def on_log(self, args, state, control, logs=None, **kw):
        if not logs or "loss" not in logs: return
        loss = logs["loss"]; self.n += 1
        self.avg = self.beta * self.avg + (1 - self.beta) * loss
        smooth = self.avg / (1 - self.beta ** self.n)
        self.lrs.append(logs["learning_rate"]); self.losses.append(smooth)
        self.best = min(self.best, smooth)
        if self.stop_div and (not math.isfinite(loss) or smooth > self.factor * self.best): control.should_training_stop = True
        return control

## Learner

`Learner` holds the data, the model, the loss, the regime and the Trainer. `learn.trainer` is the raw `transformers.Trainer`, with its arguments at `learn.args`; the verbs below change only the schedule-related arguments before each run, so anything else set there sticks. `lr` is one value for the head, the encoder getting a quarter of it (Laya's ratio: 1e-4 and 2.5e-5), or a pair `(encoder, head)`.

In [ ]:
#| export
def _lrs(lr, default_head:float=1e-4, ratio:float=0.25):
    "(encoder, head) learning rates from a head LR or a pair"
    if lr is None: lr = default_head
    return (float(lr[0]), float(lr[1])) if isinstance(lr, (tuple, list)) else (float(lr) * ratio, float(lr))

class Learner:
    "Data, model, loss and a `DecisionTrainer`, with fastai's training verbs"
    def __init__(self,
                 data:TypedDecisions, # cases, bound to the encoder's tokenizer
                 model:EncoderDecisionModel, # encoder and head, regime already set
                 loss=None, # 'soft_ce', 'rlcd', a callable, or the data's default
                 cache:str|None=None, # 'masks' or 'rows' to train a frozen encoder's head from cached features
                 lr=1e-4, # head LR, or (encoder, head)
                 preset:str|None=None, # a key of PRESETS; SYSONE_PRESET or the hardware's by default
                 path=None, # where the Trainer writes; a temporary folder by default
                 cache_dir=None, # where feature caches live; ~/.cache/sysone/features by default
                 cache_dtype:str="float16", # how a feature cache stores the vectors; float32 for an encoder whose vectors float16 cannot hold
                 calibrate_after_fit:bool=True, # fit temperatures on the calib split after every fit
                 callbacks:list|None=None, # extra TrainerCallbacks
                 act_weight:float=0.5, # the act head's loss weight, beside the option loss
                 metrics:list|None=None, # more metrics, beside Laya's: functions of a `Preds`, or `Metric`s (see metrics)
                 **args): # extra TrainingArguments
        self.data, self.model, self.cache, self.lr, self.cache_dir, self.cache_dtype = data, model, cache, lr, cache_dir, cache_dtype
        self.path = Path(path) if path else Path(tempfile.mkdtemp(prefix="sysone-"))
        self.preset = preset or default_preset()
        self.loss = get_loss(loss, data.rows("train")["target"])
        self.temperatures, self.calibrate_after_fit, self._features, self.act_threshold, self.act_score = {}, calibrate_after_fit, {}, None, None
        self.metrics, self._keys = [as_metric(m) for m in metrics or []], {}
        if cache is not None: self._check_cache(cache)
        self.recorder, self.calibration = Recorder(), CalibrationCallback(self)
        self.calibration.active = calibrate_after_fit
        self.args = training_args(self.path / "trainer", **(PRESETS[self.preset] | args))
        cbs = [SigmaSchedule(self.loss), OptionShuffle(data.builder.tfms), self.recorder, self.calibration] + list(callbacks or [])
        enc, head = _lrs(lr)
        self.trainer = DecisionTrainer(model=model, args=self.args, data_collator=data.collator, loss_fn=self.loss,
                                       compute_metrics=self._metrics, callbacks=cbs, lr_head=head, lr_encoder=enc, act_weight=act_weight)

    def __repr__(self):
        n = self.model.n_params()
        return (f"Learner({self.model.spec.id if self.model.spec else 'encoder'} · regime {self.model.regime} · "
                f"head {self.model.head.layers} layers/{self.model.head.scorer_kind} · cache {self.cache} · "
                f"loss {getattr(self.loss, '__name__', self.loss)} · {n['trainable']:,} of {n['encoder'] + n['head']:,} params train)")

In [ ]:
#| export
@patch
def _check_cache(self:Learner, mode):
    if mode not in ("masks", "rows"): raise ValueError(f"cache must be 'masks' or 'rows', not {mode!r}")
    if not self.model.frozen: raise ValueError("a feature cache needs a frozen encoder (train='head')")
    if mode == "masks" and self.model.head.layers:
        raise ValueError("cache='masks' stores anchor vectors only, which feed a head without layers; use cache='rows'")

def _split_preds(predictions):
    "(logits, p(act) or None) from what the Trainer gathered"
    logits, act = predictions if isinstance(predictions, (tuple, list)) else (predictions, None)
    return logits, (act_probability(torch.as_tensor(act)).numpy() if act is not None else None)

@patch
def _metrics(self:Learner, p) -> dict:
    t, m, q, l = unpack_labels(p.label_ids)
    logits, act = _split_preds(p.predictions)
    ans = unpack_answerable(p.label_ids)
    out = decision_metrics(np.where(m, logits, -1e4), t, m, q, l, act=act, answerable=ans)
    if self.metrics:            # the Trainer evaluates the valid split, in order: its rows say which question each one is
        meta = self.row_meta("valid") if len(t) == len(self.dataset("valid") or []) else {}
        pr = Preds(logits, t, m, q, l, answerable=ans, act=act, qids=meta.get("qid"), case_idx=meta.get("case_idx"),
                   option_keys=self.option_keys("valid") if meta else None)
        out |= {mm.name: mm(pr) for mm in self.metrics}
    return out

@patch
def dataset(self:Learner, split:str="train"):
    "What the Trainer reads for a split: cached features, rows built on read, or static rows"
    if self.data.cases.get(split) is None: return None
    if self.cache is not None:
        if split not in self._features:
            from sysone.cache import FeatureCache
            self._features[split] = FeatureCache(self.model, self.data, self.cache, cache_dir=self.cache_dir, dtype=self.cache_dtype,
                                                 batch_size=self.args.per_device_eval_batch_size).features(split)
        return self._features[split]
    if split == "train" and self.data.needs_lazy_rows: return self.data.lazy_rows(split)
    if self.data.builder.processor is not None: return self.data.lazy_rows(split)
    return self.data.rows(split)

In [ ]:
torch.manual_seed(0)
model = set_regime(EncoderDecisionModel(spec.load_encoder(), DecisionHead(spec.width, layers=1), spec), "full")
learn = Learner(data, model, preset="cpu", logging_steps=1)
learn

In [ ]:
test_eq(learn.trainer.lr_head, 1e-4); test_eq(learn.trainer.lr_encoder, 2.5e-5)
assert isinstance(learn.loss, RLCD)    # typed-decisions gold is a distribution
test_eq(len(learn.dataset("train")), len(data.rows("train")))

### Running the Trainer

`_run` is the one place the Trainer runs: it sets the schedule-related arguments on `learn.args`, resets the optimizer so new learning rates apply, points the Trainer at the right datasets, trains, and puts the arguments back. It also notes on the model when the encoder has trained, so an export after a later `freeze()` still carries the trained encoder. With a feature cache the length grouping is off (there is no sequence to pad), and so is the `n_tokens` column's role.

In [ ]:
#| export
@patch
def _run(self:Learner, epochs=None, lr=None, max_steps=None, schedule=None, eval=True, record=True, **args):
    if not self.model.head.standardizer_fitted: self.fit_standardizer()
    enc, head = _lrs(lr if lr is not None else self.lr)
    if record and any(p.requires_grad for p in self.model.encoder.parameters()):
        self.model.encoder_trained = True     # the export must then carry the encoder, whatever the regime is later
    t = self.trainer
    t.lr_head, t.lr_encoder, t.schedule, t.optimizer, t.lr_scheduler = head, enc, schedule, None, None
    kw = dict(learning_rate=head, num_train_epochs=epochs or 1, max_steps=max_steps or -1) | args
    if eval and self.data.cases["valid"] is not None: kw.setdefault("eval_strategy", "epoch")
    else: kw["eval_strategy"] = "no"
    train_ds = self.dataset("train")
    if self.cache is not None or not hasattr(train_ds, "column_names"): kw.setdefault("train_sampling_strategy", "random")
    kw = _compat(kw)
    # set the fields on the live arguments and put them back after: a new TrainingArguments object would
    # re-run its device setup, which resets accelerate's state under the Trainer's accelerator
    old = {k: getattr(t.args, k) for k in kw}
    for k, v in kw.items(): setattr(t.args, k, v)
    t.train_dataset, t.eval_dataset = train_ds, self.dataset("valid") if kw["eval_strategy"] != "no" else None
    try: out = t.train()
    finally:
        for k, v in old.items(): setattr(t.args, k, v)
    return out

A head built with `standardize=True` needs the statistics of the encoder's vectors before it trains; `_run` and `lr_find` fit them on first use, from the cached anchors when there is a cache, and otherwise from one pass of the encoder over up to `n` training rows (the anchors for a head without layers, every position for one with layers):

In [ ]:
#| export
_NOT_INPUTS = ("input_ids", "attention_mask", "marker_pos", "marker_mask", "qtype", "target", "label", "spans", "answerable", "query", "first")

@patch
def fit_standardizer(self:Learner, n:int=512, batch_size:int=8):
    "Fit the head's per-dimension standardisation on up to `n` training rows"
    head, ds = self.model.head, self.dataset("train")
    idx, queries = list(range(min(n, len(ds)))), None
    col = lambda c: [torch.as_tensor(np.asarray(ds[i][c], dtype=np.float32)) for i in idx]
    if self.cache in ("masks", "rows"):
        vecs = torch.cat(col("anchors" if self.cache == "masks" else "hidden"))
        if head.stream_query: queries = torch.stack(col("query"))
    else:
        model, out, qs, was = self.model, [], [], self.model.training
        model.eval()
        try:
            with torch.no_grad():
                for s in range(0, len(idx), batch_size):
                    b = self.data.collator([ds[j] for j in idx[s:s + batch_size]]).to(model.device)
                    o = model.encode_full(b["input_ids"], b["attention_mask"], **{k: v for k, v in b.items() if k not in _NOT_INPUTS})
                    h = o.last_hidden_state.float()
                    if head.stream_query: qs.append(model.stream_query(o).float().cpu())
                    if head.layers: out.append(h[b["attention_mask"].bool()].cpu())
                    else: out.append(head._readout(h, b["marker_pos"], b.get("spans"))[b["marker_mask"].bool()].cpu())
        finally: model.train(was)
        vecs, queries = torch.cat(out), torch.cat(qs) if qs else None
    head.fit_standardizer(vecs, query=queries)
    return self

In [ ]:
torch.manual_seed(0)
ls = Learner(data, set_regime(EncoderDecisionModel(spec.load_encoder(), DecisionHead(spec.width, layers=0, standardize=True), spec), "head"), preset="cpu", calibrate_after_fit=False)
test_eq(ls.model.head.standardizer_fitted, False)
ls._run(1, 1e-3)
test_eq(ls.model.head.standardizer_fitted, True)
test_eq(ls.model.head.in_std.min().item() > 0, True)

::: {.callout-note title="Finding: a new TrainingArguments resets the Trainer's accelerator"}
The first `_run` built a fresh `TrainingArguments` for every run. Creating one resets accelerate's global state (`TrainingArguments.__post_init__` sets up the device, which calls `AcceleratorState._reset_state()`), and the Trainer's accelerator, built from that state, is left without it: the next training step fails. `_run` therefore sets its fields on the live arguments and puts them back afterwards. What a second arguments object does to a learner:
:::

In [ ]:
TrainingArguments(tempfile.mkdtemp(), use_cpu=True, report_to="none")     # what building fresh arguments for a run does
test_fail(lambda: learn._run(1, 1e-3), contains="AcceleratorState")
learn.trainer.create_accelerator_and_postprocess()                          # rebuild the accelerator, so the notebook can go on

### Predictions and metrics

`get_preds` runs the Trainer's prediction loop over a split and returns the logits with everything the metrics need, as a [`Preds`](04a_metrics.ipynb#preds): the gold, the masks, the kinds and labels, each row's p(act) with an act head, and each row's question, case and option keys (`row_meta` and `option_keys` read them off the split's rows, in the order the Trainer reads them). `validate` turns them into the notebook's metrics, calibrated with the fitted temperatures unless told otherwise, plus, once an act threshold is fitted, the coverage and the accuracy on the cases acted on.

In [ ]:
#| export
@patch
def row_meta(self:Learner, split:str="valid") -> dict:
    "Each row's question id, case index and case id, in the order the Trainer reads the split"
    ds = self.dataset(split)
    if ds is None: raise ValueError(f"no {split!r} split")
    if hasattr(ds, "column_names"): return {"qid": [str(v) for v in ds["qid"]], "case_idx": [int(v) for v in ds["case_idx"]], "case": [str(v) for v in ds["case"]]}
    return {"qid": [x[2] for x in ds.index], "case_idx": [int(x[0]) for x in ds.index], "case": [x[3] for x in ds.index]}

@patch
def option_keys(self:Learner, split:str="valid") -> list:
    "Each row's option keys, in the order the Trainer reads a held-out split (looked up once per split)"
    if split not in self._keys:
        meta, cases, ds, recs, out = self.row_meta(split), self.data.cases[split], self.dataset(split), {}, []
        ks = [len(x) for x in ds["markers"]] if hasattr(ds, "column_names") else [None] * len(meta["qid"])
        for ci, qid, k in zip(meta["case_idx"], meta["qid"], ks):
            rec = recs.setdefault(ci, case_record(cases, ci))
            q = as_questions(rec["questions"])[qid]
            if k is not None and q.k != k:          # a transform changed the options (a shortlist): its keys are the transformed question's
                q = as_questions(apply_tfms(self.data.builder.tfms, rec)["questions"])[qid]
            out.append([str(x) for x in q.keys])
        self._keys[split] = out
    return self._keys[split]

@patch
def get_preds(self:Learner, split:str="valid") -> Preds:
    "Every row of a split as a `Preds`: logits, gold, masks, kinds and labels, p(act) with an act head, each row's question, case and option keys, and the learner's temperatures"
    ds = self.dataset(split)
    if ds is None: raise ValueError(f"no {split!r} split")
    out = self.trainer.predict(ds)
    t, m, q, l = unpack_labels(out.label_ids)
    logits, act = _split_preds(out.predictions)
    meta = self.row_meta(split)
    return Preds(logits, t, m, q, l, answerable=unpack_answerable(out.label_ids), act=act, qids=meta["qid"], case_idx=meta["case_idx"],
                 temperatures=self.temperatures, option_keys=self.option_keys(split))

def _acting_right(p) -> np.ndarray:
    "Whether acting on each row would be right: its gold is among the options and the top option is it"
    return (p["logits"].argmax(-1) == p["labels"]) & (p["answerable"] == 1)

def _act_scores(p, temperatures, how) -> tuple:
    "The scores an act threshold applies to, and their name: the act head's p(act) ('head'), or the calibrated top probability ('confidence')"
    if how == "auto": how = "head" if "act" in p else None
    if how is None: return None, None
    if how == "head":
        if "act" not in p: raise ValueError("act_score='head' needs an act head (act=True); act_score='confidence' works without one")
        return p["act"], "head"
    if how != "confidence": raise ValueError(f"act_score must be 'head', 'confidence' or 'auto', not {how!r}")
    T = np.array([lookup_temperature(temperatures, int(q), int(k)) for q, k in zip(p["qtypes"], p["masks"].sum(1))])
    z = p["logits"] / T[:, None]; z = np.exp(z - z.max(1, keepdims=True)) * p["masks"]
    return (z / z.sum(1, keepdims=True)).max(1), "confidence"

@patch
def validate(self:Learner, split:str="valid", calibrated:bool=True, metrics:list|None=None) -> dict:
    "The notebook's metrics on a split, the learner's own `metrics`, and any more given here"
    p = self.get_preds(split)
    m = decision_metrics(p["logits"], p["targets"], p["masks"], p["qtypes"], p["labels"], self.temperatures if calibrated else None,
                         act=p.get("act"), answerable=p["answerable"])
    if self.act_threshold is not None:
        s, _ = _act_scores(p, self.temperatures, self.act_score or "auto")
        if s is not None: m |= selective_metrics(s, _acting_right(p), row_thresholds(self.act_threshold, p["qtypes"], p["masks"].sum(1)))
    if not calibrated: p.temperatures = None
    return m | {mm.name: mm(p) for mm in self.metrics + [as_metric(x) for x in metrics or []]}

### calibrate

One temperature per question type and option-count bucket, fitted on the held-out calibration slice and never on training rows; the Decider divides each row's logits by its bucket's temperature. With no calib split it falls back to the validation split, with a warning, since those rows are held out too. With an act head it also fits the act thresholds on the same rows, per question type and option-count bucket as the temperatures are: the lowest p(act) at which the cases acted on are still right `act_accuracy` of the time (0.9 by default), which the Decider reports as `action.act`. `act_score="confidence"` puts the thresholds on the calibrated top probability instead, and it works for a model without an act head too. The learner remembers the choice, so the calibration after a later fit keeps it.

In [ ]:
#| export
@patch
def calibrate(self:Learner, split:str="calib", min_rows:int=10, act_accuracy:float|None=0.9, act_score:str|None=None) -> dict:
    "Fit temperatures per question type and option-count bucket on held-out rows, and act thresholds on p(act) or ('confidence') the calibrated top probability"
    if self.data.cases.get(split) is None:
        if self.data.cases.get("valid") is None: raise ValueError("calibration needs held-out rows: a calib or valid split")
        warnings.warn(f"no {split!r} split; calibrating on 'valid'"); split = "valid"
    p = self.get_preds(split)
    self.temperatures = fit_temperatures(p["logits"], p["targets"], p["masks"], p["qtypes"], min_rows)
    s, how = _act_scores(p, self.temperatures, act_score or self.act_score or "auto")
    if s is not None and act_accuracy is not None:
        self.act_threshold = fit_thresholds(s, _acting_right(p), p["qtypes"], p["masks"].sum(1), act_accuracy, min_rows)
        self.act_score = how
    return self.temperatures

### fit and fit_one_cycle

`fit` warms up and then holds the learning rate; `fit_one_cycle` warms up for `pct_start` of the steps and then follows a cosine to zero, through the Trainer's own scheduler, or torch's exact `OneCycleLR` with `onecycle=True`. Both return the last epoch's metrics; the calibration callback has fitted the temperatures by then.

In [ ]:
#| export
@patch
def _after_fit(self:Learner):
    return self.recorder.metrics[-1] if self.recorder.metrics else None

@patch
def fit(self:Learner, epochs:int, lr=None, warmup:float=0.1, **args):
    "Train for `epochs`: linear warmup, then a constant learning rate"
    self._run(epochs, lr, lr_scheduler_type="constant_with_warmup", warmup_steps=warmup, **args)
    return self._after_fit()

@patch
def fit_one_cycle(self:Learner, epochs:int, lr=None, pct_start:float=0.25, onecycle:bool=False, **args):
    "Train for `epochs` with warmup then cosine annealing (`onecycle=True`: torch's OneCycleLR)"
    if onecycle: self._run(epochs, lr, schedule=("onecycle", pct_start), **args)
    else: self._run(epochs, lr, lr_scheduler_type="cosine", warmup_steps=pct_start, **args)
    return self._after_fit()

In [ ]:
torch.manual_seed(0)
before = {k: v.clone() for k, v in learn.model.state_dict().items()}
res = learn.fit_one_cycle(2, lr=1e-3)
res

In [ ]:
test_eq(len(learn.recorder.metrics), 2)
assert all(math.isfinite(l) for l in learn.recorder.losses)
changed = [k for k, v in learn.model.state_dict().items() if not torch.equal(v, before[k])]
assert any(k.startswith("head.") for k in changed) and any(k.startswith("encoder.") for k in changed)
test_eq(set(learn.temperatures) <= {"choice", "score", "noul", "choice:3-5", "choice:2", "score:3-5", "noul:2"}, True)

The three schedules, as the Trainer and `create_scheduler` build them, for a run of 100 steps at a head learning rate of 10⁻³. Both groups follow the same shape, the encoder at a quarter of the head:

In [ ]:
#| code-fold: true
#| code-summary: figure code
def lr_curve(kind, steps=100, head=1e-3, warmup=0.25):
    "The learning rate of each group at every step, from the scheduler a run would build"
    ps = [torch.nn.Parameter(torch.zeros(1)) for _ in range(2)]
    opt = torch.optim.AdamW([{"params": [ps[0]], "lr": head}, {"params": [ps[1]], "lr": head / 4}])
    if kind == "onecycle":
        sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[head, head / 4], total_steps=steps, pct_start=warmup, anneal_strategy="cos", cycle_momentum=False)
    else: sch = get_scheduler(kind, opt, num_warmup_steps=int(warmup * steps), num_training_steps=steps)
    out = []
    for _ in range(steps): out.append([g["lr"] for g in opt.param_groups]); opt.step(); sch.step()
    return np.array(out)
curves = {"fit: constant after 10% warmup": lr_curve("constant_with_warmup", warmup=0.1),
          "fit_one_cycle: cosine after 25% warmup": lr_curve("cosine"), "fit_one_cycle(onecycle=True)": lr_curve("onecycle")}
fig, axs = plt.subplots(1, 3, figsize=(10, 2.3), sharey=True)
for ax, (name, c) in zip(axs, curves.items()):
    ax.plot(c[:, 0], label="head"); ax.plot(c[:, 1], label="encoder: a quarter"); ax.set_title(name); ax.set_xlabel("step")
axs[0].set_ylabel("learning rate"); axs[0].legend(); plt.tight_layout(); plt.show()

The metrics on the validation split, calibrated with the temperatures the fit just produced, and a recalibration with a lower row threshold (the fixture is tiny):

In [ ]:
p = learn.get_preds("valid")
test_eq(p["logits"].shape[0], len(data.rows("valid")))
test_eq(p["masks"].sum(-1).tolist(), [len(m) for m in data.rows("valid")["markers"]])
learn.validate()

::: {.callout-note title="Finding: the Trainer groups evaluation rows by length too"}
The check just above, that the predicted rows come back in the dataset's order, failed at first. With `train_sampling_strategy="group_by_length"`, which sysone uses to cut padding in training, transformers' `_get_eval_sampler` groups the evaluation rows by length as well, so `predict` returns them shuffled. The metrics computed inside the loop are unaffected, since the labels travel with the predictions, but anything that joins predictions back to rows by position (answers per case, accuracy per workflow) would be scrambled. `DecisionTrainer` keeps evaluation sequential:
:::

In [ ]:
valid_rows = learn.dataset("valid")
grouped, ours = list(Trainer._get_eval_sampler(learn.trainer, valid_rows)), list(learn.trainer._get_eval_sampler(valid_rows))
test_eq(ours, list(range(len(valid_rows))))
test_ne(grouped, ours)
grouped

In [ ]:
learn.calibrate(min_rows=5)

The temperatures come from the callback, so they are fitted however training starts, here through the raw Trainer:

In [ ]:
calls, calibrate = [], learn.calibrate
learn.calibrate = lambda *a, **k: calls.append(1) or calibrate(min_rows=4)   # the fixture's calib slice is tiny
learn.temperatures = {}
learn._run(1, 1e-3)                   # learn.trainer.train() with the schedule arguments set
del learn.calibrate
test_eq(len(calls), 1); assert learn.temperatures

The optimizer groups carry the discriminative learning rates, and the no-decay rule holds in both parts:

In [ ]:
[(g["name"], g["lr"], g["weight_decay"], len(g["params"])) for g in learn.trainer.optimizer.param_groups]

In [ ]:
g = {x["name"]: x for x in learn.trainer.optimizer.param_groups}
test_eq(sorted(g), ["encoder/decay", "encoder/no_decay", "head/decay", "head/no_decay"])
test_eq(learn.trainer.optimizer.param_groups[0]["name"], "head/decay")

### Metrics

`decision_metrics` is always computed, and `metrics=[...]` adds more columns, fastai's way: functions of a `Preds`, or `Metric`s that bind arguments, keep a metric to one kind of question or rename it (see [metrics](04a_metrics.ipynb)). The learner computes them after every epoch on the valid split, so the recorder keeps a history of each, and `validate` computes them on any split, plus any more given to it. Per-question metrics such as macro-F1 group the rows by question and count classes by option key, as `answer_metrics` does, since `get_preds` knows both.

In [ ]:
from sysone.metrics import F1Score, Metric, f1_score, log_loss, top_k_accuracy

In [ ]:
torch.manual_seed(0)
lm = Learner(data, set_regime(EncoderDecisionModel(spec.load_encoder(), DecisionHead(spec.width, layers=1), spec), "full"), preset="cpu",
             calibrate_after_fit=False, metrics=[F1Score(), Metric(f1_score, qtype="noul"), top_k_accuracy])
lm.fit(2, 1e-3)
test_eq([k for k in ("f1_score", "f1_score_noul", "top_k_accuracy") if k in lm.recorder.metrics[-1]], ["f1_score", "f1_score_noul", "top_k_accuracy"])
v, pv = lm.validate(metrics=[log_loss]), lm.get_preds()
test_close(v["f1_score"], f1_score(pv)); test_eq("log_loss" in v, True)
test_eq((len(pv.qids), len(pv.option_keys), pv.n), (len(lm.dataset("valid")),) * 3)
test_eq([pv.option_keys[i][int(pv.labels[i])] for i in range(3)],
        [str(as_questions(case_record(data.cases["valid"], c)["questions"])[q].keys[int(l)]) for c, q, l in zip(pv.case_idx[:3], pv.qids[:3], pv.labels[:3])])
{k: round(v[k], 3) for k in ("accuracy", "f1_score", "f1_score_noul", "top_k_accuracy", "log_loss")}

### Freezing

`freeze` trains the head only, `unfreeze` everything, `freeze_to(n)` all but the embeddings and the first n layers. With a feature cache configured, freezing turns it back on and unfreezing turns it off, since features cached from a frozen encoder go stale once it trains.

In [ ]:
#| export
@patch
def freeze(self:Learner):
    "Train the head only (and use the feature cache, if one is configured)"
    set_regime(self.model, "head")
    if getattr(self, "_saved_cache", None): self.cache, self._saved_cache, self._features = self._saved_cache, None, {}
    return self

@patch
def unfreeze(self:Learner):
    "Train the encoder too (its adapters, for a LoRA model); the feature cache is set aside"
    if getattr(self.model, "adapter_params", None) and hasattr(self.model.encoder, "peft_config"): thaw_adapters(self.model)
    else: set_regime(self.model, "full")
    if self.cache is not None: self._saved_cache, self.cache, self._features = self.cache, None, {}
    return self

@patch
def freeze_to(self:Learner, n:int):
    "Freeze the encoder's embeddings and first n layers"
    freeze_to(self.model, n)
    if self.cache is not None: self._saved_cache, self.cache, self._features = self.cache, None, {}
    return self

In [ ]:
learn.freeze()
test_eq(learn.model.n_params()["trainable"], learn.model.n_params()["head"])
learn.freeze_to(2)
test_eq([all(p.requires_grad for p in l.parameters()) for l in encoder_layers(learn.model.encoder)], [False, False, True, True])
learn.unfreeze()
test_eq(learn.model.frozen, False)
test_eq(getattr(learn.model, "encoder_trained", False), True)     # it trained in the fit above

### lr_find

The learning-rate range test: from `start` to `end` in `steps` exponential steps, recording the loss, stopping when it diverges. The model and optimizer state are restored afterwards, so it can run at any point, and its steps stay out of `learn.recorder`, which holds the training history. It suggests two values: `valley`, fastai's default, the middle of the longest stretch where the loss falls, and `steep`, where it falls fastest. With matplotlib installed, `plot=True` draws the curve.

In [ ]:
#| export
class SuggestedLRs(dict):
    "Learning rates suggested by `lr_find`"
    def __getattr__(self, k):
        try: return self[k]
        except KeyError: raise AttributeError(k) from None
    def __repr__(self): return "SuggestedLRs(" + ", ".join(f"{k}={v:.2e}" for k, v in self.items()) + ")"

def valley(lrs, losses):
    "fastai's valley: halfway along the longest run over which the loss keeps falling"
    n = len(losses)
    if n < 3: return float(lrs[int(np.argmin(losses))])
    max_start, max_end, lds = 0, 0, [1] * n
    for i in range(1, n):
        for j in range(i):
            if losses[i] < losses[j] and lds[i] < lds[j] + 1: lds[i] = lds[j] + 1
            if lds[max_end] < lds[i]: max_end, max_start = i, i - lds[i]
    sections = (max_end - max_start) / 3
    return float(lrs[max_start + int(sections) + int(sections / 2)])

def steep(lrs, losses):
    "The LR where the loss falls fastest against log LR"
    if len(losses) < 3: return lrs[0]
    g = np.gradient(np.asarray(losses), np.log(np.asarray(lrs)))
    return float(lrs[int(np.argmin(g))])

In [ ]:
lr_grid = list(np.logspace(-6, 0, 60))
toy = [3.0 - 2.5 / (1 + np.exp(-(np.log10(l) + 3.5) * 3)) + 8 * max(0, np.log10(l) + 1.5) ** 2 for l in lr_grid]
sv, ss = valley(lr_grid, toy), steep(lr_grid, toy)
lr_min = lr_grid[int(np.argmin(toy))]
assert 1e-5 < sv < lr_min and 1e-5 < ss < lr_min   # both on the falling side of the curve
sv, ss

Why not the minimum? At the lowest point of the curve the loss is about to blow up: the learning rate is at the edge of what training tolerates, and on a real run the smoothed curve lags behind the steps as well. The suggestions sit on the long descent before it, `steep` where the loss falls fastest and `valley` halfway along the longest stretch over which it keeps falling:

In [ ]:
#| code-fold: true
#| code-summary: figure code
fig, ax = plt.subplots(figsize=(5, 2.5))
ax.plot(lr_grid, toy, color="gray", lw=1.2); ax.set_xscale("log")
for name, v, c in (("steep", ss, "C1"), ("valley", sv, "C0"), ("minimum", lr_min, "C3")): ax.axvline(v, ls="--", color=c, lw=1, label=f"{name} {v:.1e}")
ax.set_xlabel("learning rate"); ax.set_ylabel("loss"); ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
#| export
@patch
def lr_find(self:Learner, start:float=1e-7, end:float=1.0, steps:int=100, stop_div:bool=True, plot:bool=False) -> SuggestedLRs:
    "LR range test: suggests `valley` and `steep`, and restores the model afterwards"
    if not self.model.head.standardizer_fitted: self.fit_standardizer()   # before the snapshot, which the test restores
    state = copy.deepcopy(self.model.state_dict())
    finder = LRFinder(stop_div=stop_div)
    t, calibrating, seen = self.trainer, self.calibration.active, len(self.recorder.losses)
    t.add_callback(finder); self.calibration.active = False
    try:
        self._run(epochs=1, lr=start, max_steps=steps, schedule=("range", start, end), eval=False, record=False,
                  logging_steps=1, max_grad_norm=0.0, **({"logging_nan_inf_filter": False} if "logging_nan_inf_filter" in _TA_FIELDS else {}))
    finally:
        t.remove_callback(finder); self.calibration.active = calibrating
        del self.recorder.losses[seen:], self.recorder.lrs[seen:]     # the range test is not training
        self.model.load_state_dict(state); t.optimizer, t.lr_scheduler = None, None
    lrs, losses = finder.lrs, finder.losses
    if len(losses) < steps and len(losses) > 8: lrs, losses = lrs[:-3], losses[:-3]   # stopped on divergence: drop the blow-up
    self.lr_curve = (lrs, losses)
    sugg = SuggestedLRs(valley=valley(lrs, losses), steep=steep(lrs, losses))
    if plot: self.plot_lr_find(sugg)
    return sugg

@patch
def plot_lr_find(self:Learner, sugg=None):
    "Plot the last `lr_find` curve (needs matplotlib)"
    import matplotlib.pyplot as plt
    lrs, losses = self.lr_curve
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(lrs, losses); ax.set_xscale("log"); ax.set_xlabel("learning rate"); ax.set_ylabel("loss (smoothed)")
    for (name, v), c in zip((sugg or {}).items(), ("C1", "C2", "C3", "C4")): ax.axvline(v, ls="--", lw=1, color=c, label=f"{name} {v:.1e}")
    if sugg: ax.legend()
    return fig

In [ ]:
learn.freeze()
w0, history = learn.model.head.scorer[1].weight.clone(), list(learn.recorder.losses)
sugg = learn.lr_find(start=1e-5, end=10, steps=40)
test_eq(torch.equal(learn.model.head.scorer[1].weight, w0), True)    # the model is restored
test_eq(learn.recorder.losses, history)                              # and the training history
assert 1e-5 <= sugg.valley <= 10 and len(learn.lr_curve[0]) > 5
sugg

`show_batch` prints training rows as the model reads them:

In [ ]:
#| export
@patch
def show_batch(self:Learner, n:int=2, split:str="train", max_chars:int=300):
    "Print the first `n` rows of a split with their anchors visible"
    self.data.show_batch(n, split, max_chars)

In [ ]:
learn.show_batch(1)

### Answers and exports

`decider`, `predict` and `export` hand over to `sysone.inference`, where the `Decider` and the export layout live; they import it when first called, so a learner has every verb whichever module it came from.

In [ ]:
#| export
@patch
def decider(self:Learner, **kw):
    "A `sysone.inference.Decider` over the learner's model in memory"
    from sysone.inference import Decider
    return Decider(self.model, self.data.builder, dict(self.temperatures),
                   **({"act_threshold": self.act_threshold, "act_score": self.act_score or "head"} | kw))

@patch
def predict(self:Learner, state, questions:dict, **kw) -> dict:
    "Answers in the Jev schema from the model in memory"
    return self.decider(**kw).predict(state, questions)

@patch
def export(self:Learner, path, **kw):
    "Write the model, template, transforms and temperatures to `path` (see `sysone.inference.export_learner`)"
    from sysone.inference import export_learner
    return export_learner(self, path, **kw)

## decision_learner

The one-call learner: an encoder id, a regime and a head shape (with `standardize=True`, a head that standardises the encoder's vectors first). `train` is `head`, `lora`, `mica`, `full` or a peft config. `head` is `laya` (Laya's head, two layers), `mlp` (Laya's scorer alone) or `linear` (a linear map); a feature cache implies a head without layers for `masks`. The text and multimodal applications wrap it with their own default encoders and templates.

In [ ]:
#| export
def decision_learner(data:TypedDecisions, encoder, train="head", cache:str|None=None, template=None, head:str="laya",
                     head_layers:int|None=None, init_from:str|None=None, loss=None, lr=None, preset:str|None=None,
                     dtype=None, lora:dict|None=None, standardize:bool=False, readout:str="anchor", query:str|None=None,
                     act:bool=False, pair_dim:int=256, **kw) -> Learner:
    "A `Learner` for `data` on the encoder `encoder` (a Hub id, a folder or an `EncoderSpec`), with regime `train`"
    spec = encoder if isinstance(encoder, EncoderSpec) else EncoderSpec.from_pretrained(encoder)
    template = template if template is not None else data.template if data.template is not None else spec.template
    if isinstance(template, str): template = RowTemplate.preset(template, tokenizer=spec.tokenizer)
    data.bind(spec.processor or spec.tokenizer, template, streams=list(spec.streams.values()))
    scorer = {"laya": "laya", "mlp": "laya", "linear": "linear", "gliner2": "gliner2", "pair": "pair"}[head]
    layers = head_layers if head_layers is not None else 0 if (head in ("mlp", "linear", "gliner2", "pair") or cache == "masks") else 2
    enc = spec.load_encoder(**({"dtype": dtype} if dtype is not None else {}))    # float32 unless told otherwise
    qw = None
    if query is not None and query.startswith("stream:"):
        name = query.removeprefix("stream:")
        if name not in getattr(enc, "sides", {}): raise ValueError(f"query {query!r} needs a stream encoder with a {name!r} stream")
        qw = enc.sides[name].out_width
    hd = DecisionHead(spec.width, layers, scorer, init_from=init_from, standardize=standardize, readout=readout, query=query,
                      query_width=qw, pair_dim=pair_dim, act=act)
    model = EncoderDecisionModel(enc, hd, spec)
    set_regime(model, train, **(lora or {}))
    return Learner(data, model, loss=loss, cache=cache, lr=lr if lr is not None else 1e-4, preset=preset, **kw)

In [ ]:
learn2 = decision_learner(TypedDecisions.from_json("fixtures/typed_decisions_tiny.jsonl", valid=0.25), FIX, train="lora",
                          preset="cpu", loss="soft_ce")
learn2

In [ ]:
test_eq(learn2.model.regime, "lora")
test_eq(learn2.model.head.layers, 2)
test_fail(lambda: decision_learner(data, FIX, train="full", cache="masks", preset="cpu"), contains="frozen encoder")
learn2.fit(1, lr=1e-3)
for dt in ("float16", "float32"):     # the cache stores the vectors in the dtype the learner asks for
    lc = decision_learner(data, FIX, train="head", cache="masks", preset="cpu", cache_dtype=dt, cache_dir=tempfile.mkdtemp())
    test_eq(lc.dataset("train").features["anchors"].dtype, dt)

An act head trains beside the option head. Here a quarter of the fixture's cases say that none of the options answers their first question, and the learner fits the act threshold with the temperatures:

In [ ]:
arecs = [json.loads(l) for l in Path("fixtures/typed_decisions_tiny.jsonl").read_text().splitlines()]
for r in arecs[::4]: r["gold"] = {**r["gold"], next(iter(r["questions"])): {"answerable": False}}
adata = TypedDecisions.from_records(arecs, valid=0.25, calib=0.25)
torch.manual_seed(0)
alearn = decision_learner(adata, FIX, train="head", preset="cpu", loss="soft_ce", act=True, calibrate_after_fit=False, logging_steps=1)
test_eq(sorted(set(adata.rows("train")["answerable"])), [0, 1])
alearn.fit(2, lr=1e-3)
alearn.calibrate(act_accuracy=0.5)
am = alearn.validate()
test_eq({"act_auroc", "act_brier", "n_unanswerable", "coverage", "acted_accuracy"} <= set(am), True)
test_eq(alearn.act_threshold is not None, True)
{k: am[k] for k in ("accuracy", "n", "n_unanswerable", "act_auroc", "coverage", "acted_accuracy")} | {"act_threshold": alearn.act_threshold}

The thresholds are a dict, one per question type and option-count bucket wherever the calib split has enough rows, plus `all`. `act_score="confidence"` fits them on the calibrated top probability instead, which needs no act head. Here the fixture's plain learner, which has none, reports `action` after that calibration, with its answer confidence as the score:

In [ ]:
test_eq(isinstance(alearn.act_threshold, dict) and "all" in alearn.act_threshold, True)
torch.manual_seed(0)
clearn = decision_learner(adata, FIX, train="head", preset="cpu", loss="soft_ce", calibrate_after_fit=False)
clearn.fit(1, lr=1e-3)
test_fail(lambda: clearn.calibrate(act_score="head"), contains="needs an act head")
clearn.calibrate(act_accuracy=0.5, act_score="confidence")
ca = next(iter(clearn.predict(arecs[1]["state"], arecs[1]["questions"]).values()))
test_eq((clearn.act_score, ca["action"]["act_probability"]), ("confidence", ca["answer_confidence"]))
test_eq({"coverage", "acted_accuracy"} <= set(clearn.validate()), True)
ca["action"]

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()